# Lab 01: LLM API Fundamentals and Multi-turn Context (Ollama + OpenAI)

This lab is local-first for zero API cost in class.
1. Local model access via Ollama (primary)
2. OpenAI API usage (optional)


## 0) Setup

Before running:
- Install dependencies with `uv sync`
- Create `.env` from `.env.example`
- Ensure Ollama is running and your Qwen model is installed


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")

def has_real_openai_key(value: str | None) -> bool:
    if not value or not value.strip():
        return False
    cleaned = value.strip()
    lowered = cleaned.lower()
    if lowered in {"your_openai_api_key_here", "sk-your_key_here"}:
        return False
    if lowered.startswith("your_"):
        return False
    return cleaned.startswith("sk-")

OPENAI_ENABLED = has_real_openai_key(OPENAI_API_KEY)

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:8b")

print("OPENAI_MODEL (optional):", OPENAI_MODEL)
print("OPENAI enabled:", OPENAI_ENABLED)
print("OLLAMA_BASE_URL:", OLLAMA_BASE_URL)
print("OLLAMA_MODEL:", OLLAMA_MODEL)


OPENAI_MODEL (optional): gpt-5-nano
OPENAI enabled: True
OLLAMA_BASE_URL: http://localhost:11434
OLLAMA_MODEL: qwen3:8b


## How `chat.completions.create` Works

- `model`: the model name (for example `qwen3:8b` on Ollama or `gpt-5-nano` on OpenAI).
- `messages`: conversation history passed to the model.
- `temperature`: controls randomness (lower = more deterministic).

### Message Roles
- `system`: global behavior/instructions for the assistant.
- `user`: the current user request.
- `assistant`: previous model output included as conversation context.


## 1) Local Ollama (Primary): OpenAI-Compatible Qwen Call

Use this section as the default classroom path.


In [3]:
ollama_client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL.rstrip('/')}/v1",
    api_key=os.getenv("OLLAMA_API_KEY", "ollama"),
)

ollama_response = ollama_client.chat.completions.create(
    model="qwen3.5:latest",
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "Explain what a token is in LLMs in 2 sentences."}
    ],
    temperature=1,
)

display(Markdown("## Qwen Response\n\n" + ollama_response.choices[0].message.content))


## Qwen Response

A token is the basic unit of text that a large language model processes, often representing a whole word, a part of a word, or punctuation. The model converts human-readable text into a sequence of these discrete tokens to perform calculations and generate predictions.

## 2) OpenAI API (Optional)

Use this only if you want to spend on hosted/premium models.


In [19]:
if not OPENAI_ENABLED:
    display(Markdown("**Skipped:** set a real `OPENAI_API_KEY` in `.env` to run this section."))
else:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    openai_response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a concise teaching assistant."},
            {"role": "user", "content": "Explain what a token is in LLMs in 2 sentences."}
        ],
        temperature=1,
    )
    display(Markdown("## OpenAI Response\n\n" + openai_response.choices[0].message.content))


**Skipped:** set a real `OPENAI_API_KEY` in `.env` to run this section.

## 3) Compare Responses

Use the same prompt and compare local vs cloud output quality, style, and latency.


In [25]:
shared_prompt = "Give 3 practical tips to write better prompts for beginner LLM users."

ollama_compare = ollama_client.chat.completions.create(
    model="qwen3.5:latest",
    messages=[{"role": "user", "content": shared_prompt}],
    temperature=1,
)

display(Markdown("## Ollama (Qwen)\n\n" + ollama_compare.choices[0].message.content))

if OPENAI_ENABLED:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    openai_compare = openai_client.chat.completions.create(
        model="qwen3.5:latest",
        messages=[{"role": "user", "content": shared_prompt}],
        temperature=1,
    )
    display(Markdown("## OpenAI\n\n" + openai_compare.choices[0].message.content))
else:
    display(Markdown("**OpenAI compare skipped** because `OPENAI_API_KEY` is not configured."))


## Ollama (Qwen)

Here are 3 practical tips to write better prompts, even as a beginner:

### 1. Assign a Role and Audience
Instead of just stating *what* you want, tell the AI *who* it should be and *who* is reading the output. This helps the model shift its tone, vocabulary, and depth to match your needs.
*   **Instead of:** "Write a recipe for meatballs."
*   **Try this:** "Act as an experienced Italian grandmother. Write a recipe for meatballs that is perfect for dinner guests. Explain the steps simply for a beginner cook."
*   **Why it works:** Giving the AI a persona ("Grandmother") and a specific audience ("Beginner cook") drastically improves the quality and tone of the response.

### 2. Define Clear Constraints and Format
AI models love to ramble. Tell them exactly how you want the information structured so you don't have to spend time editing the output. Use constraints like word counts, bullet points, or specific formatting.
*   **Instead of:** "Tell me about the history of coffee."
*   **Try this:** "Summarize the history of coffee in 5 lines. Use bullet points for key facts. Do not include any dates prior to 1500."
*   **Why it works:** This forces the AI to be concise and ensures the response is immediately usable (e.g., ready for a presentation slide or email).

### 3. Provide Context or Background Information
Beginners often assume the AI knows the situation, but it doesn't. The more background you give, the less generic the answer. Explain the *why* behind your request.
*   **Instead of:** "Write a job interview question for a software engineer."
*   **Try this:** "We are hiring a junior software engineer at a startup. We need to test their problem-solving skills, not just their knowledge. Write a behavioral interview question that asks them about a past failure."
*   **Why it works:** This prevents the AI from giving you a generic, high-level corporate response. The AI adapts its answer based on the scenario you describe.

**OpenAI compare skipped** because `OPENAI_API_KEY` is not configured.

## 4) Multi-turn Example: Feed Assistant Output Back As Context

This demonstrates how an assistant message from turn 1 is sent back in turn 2 using `role="assistant"`.


In [ ]:
seed_topic = "vector databases in RAG"

turn1 = ollama_client.chat.completions.create(
    model="qwen3.5:latest",
    messages=[
        {"role": "system", "content": "You are a helpful instructor."},
        {"role": "user", "content": f"Generate one strong interview question about {seed_topic}. Return the question only"},
    ],
    temperature=1,
)

assistant_question = turn1.choices[0].message.content
display(Markdown("## Turn 1: Assistant-Generated Question\n\n" + assistant_question))

turn2 = ollama_client.chat.completions.create(
    model="qwen3.5:latest",
    messages=[
        {"role": "system", "content": "You are a helpful instructor."},
        {"role": "assistant", "content": assistant_question},
        {"role": "user", "content": "Now answer that question in a concise beginner-friendly way."},
    ],
    temperature=1,
)

display(Markdown("## Turn 2: Answer Using Assistant Context\n\n" + turn2.choices[0].message.content))


## Turn 1: Assistant-Generated Question

How do you balance the trade-offs between retrieval accuracy and query latency when implementing hybrid search or complex metadata filtering in a production RAG system that relies on a vector database?

## Turn 2: Answer Using Assistant Context

Think of retrieval like searching for a book in a library.

**The Trade-off:**
*   **Accuracy:** To find the exact right book, the librarian (your system) has to read the details on every spine (metadata) and check the contents (vectors). This takes time.
*   **Latency:** The faster you are, the less likely you are to read details, which might mean missing the right book.

**How to Balance It:**
1.  **Start Simple:** Begin with basic keyword searches. Add complex filters (like date, source, or tags) only if the simple search isn't giving good results.
2.  **Filter Sparingly:** Every extra rule you add slows down the system. Use only the most important filters.
3.  **Set a Time Limit:** In production, users expect answers in under 2 seconds. If adding a filter breaks that speed, simplify it.

**The Goal:** Find the "sweet spot" where you give correct answers without frustrating users with long wait times.

## 5) Multi-turn Example (OpenAI API, Optional)

Same pattern as above, but using OpenAI API. This is optional and may incur cost.


In [ ]:
if not OPENAI_ENABLED:
    display(Markdown("**Skipped:** set a real `OPENAI_API_KEY` in `.env` to run this section."))
else:
    seed_topic = "function calling in agents"
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    openai_turn1 = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful instructor."},
            {"role": "user", "content": f"Generate one strong interview question about {seed_topic}. Return the question only"},
        ],
        temperature=1,
    )

    openai_assistant_question = openai_turn1.choices[0].message.content
    display(Markdown("## OpenAI Turn 1: Assistant-Generated Question\n\n" + openai_assistant_question))

    openai_turn2 = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful instructor."},
            {"role": "assistant", "content": openai_assistant_question},
            {"role": "user", "content": "Now answer that question in 4 short bullet points for beginners."},
        ],
        temperature=1,
    )

    display(Markdown("## OpenAI Turn 2: Answer Using Assistant Context\n\n" + openai_turn2.choices[0].message.content))


**Skipped:** set a real `OPENAI_API_KEY` in `.env` to run this section.

## 6) Exercises
- Build your own 3-turn loop: (1) model generates a question, (2) you pass that as an `assistant` message, (3) model answers and then proposes one follow-up question.
- Repeat the loop for 2 more turns while keeping the full `messages` history.
- Compare how the conversation quality changes when you remove prior `assistant` messages.
- Optional: run the same loop on both Ollama and OpenAI and compare results.


**Hint (persistent message history):** keep a single `messages` list, append each new `assistant` output and `user` prompt, then pass the full list again in the next call.


## 6) Exercises Part 1: Build your own 3-turn Loop

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful instructor teaching beginners about AI."},
    {"role": "user", "content": "Generate a beginner question about artificial intelligence."}
]

for i in range(3):

    response = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        temperature=1,
    )

    assistant_output = response.choices[0].message.content

    display(Markdown(f"### Turn {i+1} Assistant Output\n\n{assistant_output}"))

    # append assistant message
    messages.append({"role": "assistant", "content": assistant_output})

    # next user instruction
    messages.append({
        "role": "user",
        "content": "Answer the previous question and propose one follow-up question."
    })

### Turn 1 Assistant Output

"Think about the last time you opened an app like Spotify or Netflix, and it suggested a song or movie that seemed like exactly what you wanted to watch. Do you think the computer there was just 'guessing,' or do you think it actually **learned** from patterns of what other people have watched or listened to? What would have to happen for an AI program to get that 'guess' right?"

### Turn 2 Assistant Output

That is a fantastic question! You actually have the core concept pretty right. The short answer is: **It is learning, not guessing.**

Here is how it works simply:

Think of the computer like a new student walking into a library. If you tell the computer, *"Look at all the books that are on the mystery shelf"* and *"'Sci-Fi' is on the same shelf as 'Thriller'*, the computer learns that connection. It doesn't have to be told for *one specific* person what they like; it looks at the "study notes" left behind by millions of other people.

It builds a list of patterns, like:
*   *"People who listen to Jazz also like Piano Music."*
*   *"People who stop watching at 11:00 PM often go back to sleep after watching an Action Movie."*

When you ask for a suggestion, it checks its memory (the patterns) and makes a **math-based prediction** instead of a random guess. That’s why it feels like magic, but it’s really just advanced pattern recognition!

***

**Follow-up Question:**
"If the computer learns from millions of different choices to make those patterns, what happens if everyone else in the world mostly likes the same kind of movies as you? Could the computer ever get stuck suggesting only the same old things forever?"

### Turn 3 Assistant Output

**Answer:**

You have hit on one of the biggest challenges in AI today! **Yes, the computer can definitely get stuck.**

When people all over the world have similar tastes, the AI sees a pattern that says, *"Oh, the best way to keep people happy is to give them exactly what most people are already watching."* This is often called an **"echo chamber"** or a **"filter bubble."**

Imagine the AI is like a popular party host. If everyone at the party loves pizza, the host decides *"Pizza is the safest choice to keep everyone happy."* Eventually, they might stop bringing cupcakes, fruits, or tacos, because they have learned that pizza always gets a high score. You might end up missing out on trying new foods because the AI thinks, *"Why would you want to risk a bad rating?"*

However, this isn't permanent! That is actually why it is important for humans to interact with the AI.

***

**Follow-up Question:**

**"If the computer wants to be safe and stick to what is popular to avoid mistakes, how would you convince the computer that you are willing to try something unexpected? What if you clicked 'Not Interested' on a suggestion—does it learn from that 'dislike' to try something new?"**

## 6) Exercises Part 2: Comapare and have a Conversation without assistant Messages

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful instructor teaching beginners about AI."},
    {"role": "user", "content": "Generate a beginner question about artificial intelligence."}
]

for i in range(3):

    response = ollama_client.chat.completions.create(
        model="qwen3.5:latest",
        messages=messages,
        temperature=1,
    )

    assistant_output = response.choices[0].message.content

    display(Markdown(f"### Turn {i+1} Assistant Output\n\n{assistant_output}"))

    # only keep user prompts
    messages.append({
        "role": "user",
        "content": "Answer the previous question and propose one follow-up question."
    })

### Turn 1 Assistant Output

"Think about your everyday life and the apps on your phone: Can you name one app or tool that makes a suggestion for you, like recommending a movie or telling you how to get somewhere? What do you think helps it make those suggestions?"

KeyboardInterrupt: 